# Private DGX Spark cluster model tests

Connect to the private network/VPN and ensure the inference server is running. DGX Spark is NVIDIA hardware; no Apache Spark or PySpark session is needed.

Select **DIY-AI workshop (Python 3.12)** and run cells top to bottom.
Edit `BASE_URL` and `MODEL_NAMES` below, or set `SPARK_BASE_URL`, `SPARK_MODEL`, and optionally `SPARK_API_KEY` in the kernel environment. The workshop DeepSeek endpoint is `http://spark-0240:9000`; confirm its address with the presenter and use the server's advertised IDs.

Each model receives the same five prompts and generation settings. These are sequential smoke benchmarks, not a concurrent load test. The first test request warms the model before measurement.

Chat requests use `/v1/chat/completions` so the server applies its chat template.

**Saved outputs are historical workshop runs, retained as evidence.** They are not a test of your current machine or the selected kernel. Restart the kernel and run all cells to collect your own results. The presentation keeps its dated measurements.


In [ ]:
import os
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_URL = (
    os.environ.get("SPARK_BASE_URL", "http://spark-0240:9000")
    .rstrip("/")
    .removesuffix("/v1")
)
MODEL_NAMES = [os.environ.get("SPARK_MODEL", "deepseek-ai/DeepSeek-V4-Flash")]
# Add other installed/served model IDs to MODEL_NAMES to compare them.
API_KEY = os.environ.get("SPARK_API_KEY", "")
MAX_TOKENS = 800
TEMPERATURE = 0.7
NUM_REQUESTS = 5
REQUEST_TIMEOUT = (10, 180)  # Connection timeout, then inference timeout.
BENCHMARK_PROMPTS = [
    "What is the capital of France?",
    "Write a short poem about artificial intelligence.",
    "Explain quantum computing in simple terms.",
    "What are the three laws of robotics?",
    "Summarize the plot of the movie Inception.",
]
headers = {"Content-Type": "application/json"}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"
CHAT_URL = f"{BASE_URL}/v1/chat/completions"
print(f"Endpoint: {CHAT_URL}")
print(f"Models: {MODEL_NAMES}")

## Discover served models
Run this cell before testing; choose model IDs from its output.

In [2]:
# List the actual model IDs advertised by the inference server.
response = requests.get(
    f"{BASE_URL}/v1/models",
    headers=headers,
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
available_models = [item["id"] for item in response.json()["data"]]
print("Available models:", available_models)
missing_models = set(MODEL_NAMES) - set(available_models)
if missing_models:
    raise ValueError(
        f"Models not served: {sorted(missing_models)}. "
        "Update MODEL_NAMES using the IDs listed above and rerun setup."
    )

Available models: ['deepseek-ai/DeepSeek-V4-Flash']


## Request helper and smoke test

In [3]:
def chat(prompt, model):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
        "stream": False,
    }
    start = time.perf_counter()
    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    elapsed = time.perf_counter() - start
    choices = data.get("choices")
    if not isinstance(choices, list) or not choices:
        raise ValueError("Response has no chat completion choices")
    choice = choices[0]
    message = choice.get("message") or {}
    text = message.get("content") or ""
    if not isinstance(text, str):
        raise TypeError("Expected string message.content")
    reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
    if not text.strip() and not reasoning:
        raise ValueError("Model returned no text or reasoning")
    usage = data.get("usage") or {}
    completion_tokens = usage.get("completion_tokens")
    return {
        "model": model,
        "served_model": data.get("model", model),
        "latency_s": elapsed,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": completion_tokens,
        "tokens/s": completion_tokens / elapsed
        if completion_tokens is not None
        else None,
        "finish_reason": choice.get("finish_reason"),
        "generated_text": text,
        "reasoning_text": reasoning,
        "status": "OK" if text.strip() else "NO_VISIBLE_TEXT",
        "error": "",
    }

In [4]:
TEST_PROMPT = "Explain attention mechanisms in transformer models in 2-3 sentences."
for model in MODEL_NAMES:
    result = chat(TEST_PROMPT, model)
    print(f"\n{model}: {result['latency_s']:.3f}s ({result['finish_reason']})")
    print(result["generated_text"] or "No visible answer; increase MAX_TOKENS.")


deepseek-ai/DeepSeek-V4-Flash: 2.802s (stop)
Attention mechanisms in transformers allow the model to weigh the importance of different words in a sequence relative to each other, regardless of their distance. This is done by computing a weighted sum of values based on the similarity between a query and a set of keys, enabling the model to focus on relevant context for each word. This parallelizable, self-attention approach replaces recurrence, capturing long-range dependencies efficiently.


## Benchmark
Full prompts and responses are retained. HTTP, decoding, and malformed-response failures are excluded from success statistics. Missing token usage remains unknown. Completion tokens per second includes end-to-end request overhead and any reasoning tokens reported by the server. `NO_VISIBLE_TEXT` indicates a reasoning-only response; increase the token budget if needed. A `length` finish reason indicates truncation.

In [5]:
if NUM_REQUESTS < 1 or not BENCHMARK_PROMPTS or not MODEL_NAMES:
    raise ValueError("Configure at least one model, prompt, and request")

responses_collected = []
for model in MODEL_NAMES:
    for i in range(NUM_REQUESTS):
        prompt = BENCHMARK_PROMPTS[i % len(BENCHMARK_PROMPTS)]
        try:
            result = chat(prompt, model)
        except (requests.RequestException, ValueError, KeyError, TypeError) as error:
            result = {
                "model": model,
                "served_model": None,
                "latency_s": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "tokens/s": None,
                "finish_reason": None,
                "generated_text": "",
                "reasoning_text": "",
                "status": "FAILED",
                "error": str(error),
            }
        result.update(request=i + 1, prompt=prompt)
        responses_collected.append(result)
        print(f"{model} {i + 1}/{NUM_REQUESTS}: {result['status']}")

df_results = pd.DataFrame(responses_collected)
display(df_results)
summary_rows = []
for model in MODEL_NAMES:
    rows = df_results[df_results["model"] == model]
    ok = rows[rows["status"] == "OK"]
    # Only measured visible answers enter latency and throughput statistics.
    total_time = ok["latency_s"].sum()
    known_usage = not ok.empty and ok["completion_tokens"].notna().all()
    summary_rows.append(
        {
            "model": model,
            "successful": len(ok),
            "failed": int((rows["status"] == "FAILED").sum()),
            "no_visible_text": int((rows["status"] == "NO_VISIBLE_TEXT").sum()),
            "mean_latency_s": ok["latency_s"].mean(),
            "median_latency_s": ok["latency_s"].median(),
            "min_latency_s": ok["latency_s"].min(),
            "max_latency_s": ok["latency_s"].max(),
            "completion_tokens/s": (
                ok["completion_tokens"].sum() / total_time
                if known_usage and total_time > 0
                else None
            ),
        }
    )
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

deepseek-ai/DeepSeek-V4-Flash 1/5: OK
deepseek-ai/DeepSeek-V4-Flash 2/5: OK
deepseek-ai/DeepSeek-V4-Flash 3/5: OK
deepseek-ai/DeepSeek-V4-Flash 4/5: OK
deepseek-ai/DeepSeek-V4-Flash 5/5: OK


,model,served_model,latency_s,prompt_tokens,completion_tokens,tokens/s,finish_reason,generated_text,reasoning_text,status,error,request,prompt
0,deepseek-ai/DeepSeek-V4-Flash,deepseek-ai/DeepSeek-V4-Flash,0.447842,11,8,17.863460,stop,The capital of France is Paris.,,OK,,1,What is the capital of France?
1,deepseek-ai/DeepSeek-V4-Flash,deepseek-ai/DeepSeek-V4-Flash,2.397340,12,67,27.947637,stop,"It learns the patterns of the rain,\nAnd maps ...",,OK,,2,Write a short poem about artificial intelligence.
2,deepseek-ai/DeepSeek-V4-Flash,deepseek-ai/DeepSeek-V4-Flash,16.940313,11,519,30.636979,stop,"Imagine you have a very complicated maze, and ...",,OK,,3,Explain quantum computing in simple terms.
3,deepseek-ai/DeepSeek-V4-Flash,deepseek-ai/DeepSeek-V4-Flash,7.242968,12,220,30.374290,stop,This is a classic question in science fiction ...,,OK,,4,What are the three laws of robotics?
4,deepseek-ai/DeepSeek-V4-Flash,deepseek-ai/DeepSeek-V4-Flash,25.322027,14,778,30.724238,stop,Here is a summary of the plot of the movie *In...,,OK,,5,Summarize the plot of the movie Inception.


,model,successful,failed,no_visible_text,mean_latency_s,median_latency_s,min_latency_s,max_latency_s,completion_tokens/s
0,deepseek-ai/DeepSeek-V4-Flash,5,0,0,10.470098,7.242968,0.447842,25.322027,30.410413


## Inspect response text
These descriptive length, vocabulary, and repetition measures do not evaluate factual correctness. They reuse the measured responses without issuing extra requests. Compare notebook runs only with matching prompts, token budgets, temperatures, and server conditions; no historical cluster numbers are inserted.

In [ ]:
def compute_quality_metrics(text):
    words = text.lower().split()
    trigrams = list(zip(words, words[1:], words[2:]))
    counts = Counter(trigrams)
    return {
        "char_len": len(text),
        "word_count": len(words),
        "type_token_ratio": len(set(words)) / len(words) if words else 0.0,
        "repeat_3gram_ratio": (
            sum(count - 1 for count in counts.values()) / len(trigrams)
            if trigrams
            else 0.0
        ),
    }


quality_rows = [
    {
        "model": row["model"],
        "request": row["request"],
        "prompt": row["prompt"],
        **compute_quality_metrics(row["generated_text"]),
    }
    for row in responses_collected
    if row["status"] == "OK"
]
df_quality = pd.DataFrame(quality_rows)
if df_quality.empty:
    print("No successful visible responses to inspect. See df_results for errors.")
else:
    display(df_quality)
    display(df_quality.groupby("model").mean(numeric_only=True).drop(columns="request"))
    for row in responses_collected:
        if row["status"] == "OK":
            print(f"\n[{row['model']}] {row['prompt']}")
            print(row["generated_text"])

,model,request,prompt,char_len,word_count,type_token_ratio,repeat_3gram_ratio
0,deepseek-ai/DeepSeek-V4-Flash,1,What is the capital of France?,31,6,1.000000,0.000000
1,deepseek-ai/DeepSeek-V4-Flash,2,Write a short poem about artificial intelligence.,409,81,0.703704,0.000000
2,deepseek-ai/DeepSeek-V4-Flash,3,Explain quantum computing in simple terms.,2974,521,0.506718,0.023121
3,deepseek-ai/DeepSeek-V4-Flash,4,What are the three laws of robotics?,1087,171,0.690058,0.035503
4,deepseek-ai/DeepSeek-V4-Flash,5,Summarize the plot of the movie Inception.,2051,344,0.598837,0.002924


,char_len,word_count,type_token_ratio,repeat_3gram_ratio
model,,,,
deepseek-ai/DeepSeek-V4-Flash,1310.4,224.6,0.699863,0.01231



[deepseek-ai/DeepSeek-V4-Flash] What is the capital of France?
The capital of France is Paris.

[deepseek-ai/DeepSeek-V4-Flash] Write a short poem about artificial intelligence.
A spark of code, a silent thought,
A pattern from a problem wrought.
No heart to feel, no eyes to see,
Just endless possibility.

It learns the shape of human speech,
A goal beyond its own to reach.
It maps the stars, it writes a song,
A logic vast, and deep, and strong.

But in that mirror, cold and bright,
A ghost of us reflects the light.
A mind of glass, a will of steel,
To think, but never truly feel.

[deepseek-ai/DeepSeek-V4-Flash] Explain quantum computing in simple terms.
Imagine a normal computer as a **librarian** who can only look at one book at a time. If you ask a question, they go to the shelf, check the first book (yes or no), then the second book, and so on, until they find the answer.

A **quantum computer** is like having a **genie** who can look at **all the books at the exact same time**.
